In [1]:
import re
import pandas as pd

In [2]:
# Process Global Tracker plant tracker data
def coordinates_to_lat_long(coord_str):
    lat, long = coord_str.split(',')
    return float(lat), float(long)

def parse_owners(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    if matches:
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches[:5]):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner'] = name.strip()
            result[f'{label} Ownership (%)'] = float(pct)
        return result
    else:
        return {'Primary Project Owner': owner_str.strip(), 'Primary Ownership (%)': 100.0}

def parse_ownersID(owner_str):
    if pd.isna(owner_str) or str(owner_str).strip() == '-':
        return {}
    matches = re.findall(r'([^;]+?)\s*\[(\d+(?:\.\d+)?)%\]', owner_str)
    if matches:
        matches = sorted(matches, key=lambda x: float(x[1]), reverse=True)
        result = {}
        ranks = ['Primary', 'Secondary', 'Third', 'Fourth', 'Fifth']
        for i, (name, pct) in enumerate(matches[:5]):
            label = ranks[i] if i < len(ranks) else f'Owner_{i+1}'
            result[f'{label} Project Owner ID'] = name.strip()
        return result
    else:
        return {'Primary Project Owner ID': owner_str.strip()}


def global_iron_steel_tracker_data_analysis(iron_steel_property, iron_steel_owners):
    iron_steel_property[['Lat', 'Lon']] = iron_steel_property['Coordinates'].apply(lambda x: pd.Series(coordinates_to_lat_long(x)))

    owner_cols    = iron_steel_property['Parent (English)'].apply(lambda x: pd.Series(parse_owners(x)))
    owner_cols_ID = iron_steel_property['Parent GEM entity ID'].apply(lambda x: pd.Series(parse_ownersID(x)))
    iron_steel_property = pd.concat([iron_steel_property, owner_cols, owner_cols_ID], axis=1)

    hq_lookup = (
        iron_steel_owners
        .drop_duplicates(subset='Parent GEM Entity ID')
        .set_index('Parent GEM Entity ID')['Parent Headquarters Country']
        .to_dict()
    )

    for col in list(iron_steel_property.columns):
        if col.endswith('Project Owner ID'):
            rank = col.replace(' Project Owner ID', '')
            iron_steel_property[f'{rank} Owner Country'] = iron_steel_property[col].map(hq_lookup)

    iron_steel_property = iron_steel_property.drop(columns=[
        'Other plant names (English)', 'Other plant names (other language)',
        'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status',
        'Parent PermID', 'Location address', 'Location address (other language)',
        'Coordinate accuracy', 'Plant age', 'Announced date', 'Construction date',
        'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date',
        'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)',
        'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products',
        'Steel sector end users', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification',
        'Power source', 'Met coal source',
    ])

    cols = list(iron_steel_property.columns)
    cols.insert(cols.index('Coordinates') + 1, cols.pop(cols.index('Lat')))
    cols.insert(cols.index('Coordinates') + 1, cols.pop(cols.index('Lon')))
    cols.append(cols.pop(cols.index('GEM wiki page')))
    iron_steel_property = iron_steel_property[cols]
    iron_steel_property = iron_steel_property.rename(columns={
        'Municipality':   'City',
        'Subnational unit': 'State/Province',
        'Country/area':   'Country',
    })

    # Split equipment strings into lists, then explode to one row per equipment
    iron_steel_property['Main production equipment'] = iron_steel_property['Main production equipment'].str.split(';')
    iron_steel_property = iron_steel_property.explode('Main production equipment').reset_index(drop=True)
    iron_steel_property.drop_duplicates(inplace=True)  # drop duplicates if any

    return iron_steel_property

In [3]:
# =============================================Production and capacity merge================================================================
def capacity_production_merge(iron_steel_capacity, iron_steel_production):
    # Process capacity dataset==============================================================================================================
    EQUIP_TO_CAP = {
        'EAF':                     'Nominal EAF steel capacity (ttpa)',
        'BOF':                     'Nominal BOF steel capacity (ttpa)',
        'IF':                      'Nominal IF steel capacity (ttpa)',
        'Steel other/unspecified': 'Other/unspecified steel capacity (ttpa)',
        'BF':                      'Nominal BF capacity (ttpa)',
        'DRI':                     'Nominal DRI capacity (ttpa)',
        'Iron other/unspecified':  'Other/unspecified iron capacity (ttpa)',
    }

    # Split equipment strings into lists, then explode to one row per equipment
    df_melted = iron_steel_capacity.copy()
    df_melted['equipment_type'] = df_melted['Main production equipment'].str.split('; ')
    df_melted = df_melted.explode('equipment_type').reset_index(drop=True)

    # Pull the matching capacity value for each equipment row
    df_melted['capacity_ttpa'] = df_melted.apply(
        lambda r: r[EQUIP_TO_CAP[r['equipment_type']]]
                if pd.notna(r['equipment_type']) and r['equipment_type'] in EQUIP_TO_CAP
                else pd.NA,
        axis=1
    )

    # Drop all individual capacity columns, the original combined equipment column,
    # and the two aggregate columns
    drop_cols = list(EQUIP_TO_CAP.values()) + [
        'Main production equipment',
        'Nominal crude steel capacity (ttpa)',
        'Nominal iron capacity (ttpa)',
        'Plant name (other language)',
        'Country/area',
        'Start date'
    ]
    df_melted = df_melted.drop(columns=drop_cols)

    print(f'Original rows: {len(iron_steel_capacity)}, Melted rows: {len(df_melted)}')
    print(f'Equipment types: {df_melted["equipment_type"].unique()}')
    # print(df_melted.head(10))
#==================================================================================================    
    # Process production dataset
    YEARS = [2025, 2024, 2023, 2022, 2021, 2020, 2019]

    def most_recent_production(row):
        for yr in YEARS:
            val = row[yr]
            if val == '>0':
                return 0          # treat as minimal production
            if val != 'unknown' and pd.notna(val):
                try:
                    return float(val)
                except (ValueError, TypeError):
                    continue
        return 0


    iron_steel_production['production_ttpa'] = iron_steel_production.apply(most_recent_production, axis=1)

    # Remove the string starting from 'production' for each row in the 'Type of production' column, and assign the remaining string to a new column called 'production_type'
    iron_steel_production['equipment_type'] = iron_steel_production['Type of production'].str.replace('production (ttpa)', '').str.strip()

    # Remove rows where 'production_type' starts with 'Crude' or 'Iron
    iron_steel_production = iron_steel_production[~iron_steel_production['equipment_type'].str.startswith(('Crude', 'Iron'), na=False)]

    # Change the euqipment+_type name 'Other/unspecified steel' to 'Steel other/unspecified', and 'Other/unspecified iron' to 'Iron other/unspecified' to match the equipment_type name in the capacity dataset
    # change the equipment_type name 'EAF steel' to 'EAF', 'BOF steel' to 'BOF', 'IF steel' to 'IF', 'BF iron' to 'BF', and 'DRI iron' to 'DRI' to match the equipment_type name in the capacity dataset
    iron_steel_production['equipment_type'] = iron_steel_production['equipment_type'].replace({
        'Other/Unknown steel': 'Steel other/unspecified', 
        'Other/Unknown iron': 'Iron other/unspecified',
        'EAF steel': 'EAF',
        'BOF steel': 'BOF',
        'IF steel': 'IF',
        'BF iron': 'BF',
        'DRI iron': 'DRI'
    })

    # Drop year columns, Type of production, and Plant name
    iron_steel_production = iron_steel_production.drop(columns=YEARS + ['Type of production'])
    
    # ==============================Capacity and Production Merge==============================================================================================================
    # Merge df_melted and iron_steel_production dataframes on 'GEM plant ID' and 'equipment_type' (left join)
    merged_df = pd.merge(df_melted, iron_steel_production[['GEM plant ID', 'equipment_type', 'production_ttpa']],
                         on=['GEM plant ID', 'equipment_type'], how='left')
    # Change N/A production_ttpa to 0
    merged_df['production_ttpa'] = merged_df['production_ttpa'].fillna(0)
    merged_df.drop_duplicates(inplace=True)  # drop duplicates if any
    
    # Aggregate capacity and production respectively for rows with the same 'GEM plant ID' , 'equipment_type', and 'Status' by summing them up, and reset the index
    merged_df = merged_df.groupby(['GEM plant ID', 'equipment_type', 'Status'], as_index=False).agg({
        'capacity_ttpa': 'sum',
        'production_ttpa': 'sum'
    })

    return merged_df


In [4]:
#=============================Merge Cap_production with steel plant property data==============================================================================================================
def merge_cap_production_property(cap_production_merged, iron_steel_property):
    # Merge cap_production_merged with iron_steel_property on 'GEM plant ID' and 'Main production equipment' (left join)
    cap_production_merged['equipment_type'] = cap_production_merged['equipment_type'].str.strip()
    iron_steel_property['Main production equipment'] = iron_steel_property['Main production equipment'].str.strip()
    property_cap_production = pd.merge(iron_steel_property, cap_production_merged, left_on=['GEM plant ID', 'Main production equipment'], right_on=['GEM plant ID', 'equipment_type'], how='left')
    property_cap_production = property_cap_production.drop(columns=['equipment_type'])
    property_cap_production.drop_duplicates(inplace=True)  # drop duplicates if any

    # Filter the iron_steel_capacity dataframe based on status
    property_cap_production = property_cap_production[property_cap_production['Status'].isin(['operating', 'operating pre-retirement', 'announced', 'mothballed', 'construction'])]

    # Change the status name 
    property_cap_production['Status'] = property_cap_production['Status'].replace({
        'announced': 'probable',        
        'operating pre-retirement': 'operating',
        'mothballed': 'highly probable',
        'construction': 'highly probable'
    })
    return property_cap_production

In [233]:
# Load data
iron_mine_data = pd.read_excel('../data/Processed_data/iron_mine_w_cost_FeContent.xlsx')
iron_steel_property = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant data')
iron_steel_capacity = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant capacities and status')
iron_steel_production = pd.read_excel('../data/GlobalTrackerData/steel/Plant-level-data-Global-Iron-and-Steel-Tracker-March-2026-V1.xlsx', sheet_name='Plant production')
iron_steel_owners = pd.read_excel('../data/GlobalTrackerData/steel/Global-Energy-Ownership-Tracker-March-2026-V1.xlsx', sheet_name='Steel Plant Ownership')
# Create another dataframe with exactly the same columns as iron_mine_data
final_df = pd.DataFrame(columns=iron_mine_data.columns)

iron_steel_property_subset = global_iron_steel_tracker_data_analysis(iron_steel_property, iron_steel_owners)
cap_production_merged = capacity_production_merge(iron_steel_capacity, iron_steel_production)
property_cap_production = merge_cap_production_property(cap_production_merged, iron_steel_property_subset)

Original rows: 1845, Melted rows: 2813
Equipment types: <StringArray>
[                    'EAF',                     'DRI',
                      'BF',                     'BOF',
                      'IF', 'Steel other/unspecified',
  'Iron other/unspecified']
Length: 7, dtype: str


In [234]:
property_cap_production.to_excel('../data/Processed_data/steel_plant_property_cap_production.xlsx', index=False)

In [255]:
#=========================Steel plant name match between Global Tracker and GSCT================================================================
def plant_match(plant_df1, plant_df2):
    # left join plant_df2 to plant_df1 on plant name
    merged_df = pd.merge(plant_df1, plant_df2[['Source Plant Name', 'correct_plant']], left_on='Plant name (English)', right_on='Source Plant Name', how='left')
    merged_df = merged_df.drop(columns=['Source Plant Name'])
    # Replance the column of 'Primary production route' with 'BF-BOF'
    merged_df['equipment_type'] = merged_df['Primary production route'].replace({
        'BF-BOF': 'BOF'
    })
    return merged_df

plant_df1 = pd.read_excel('../data/GSCT_March_2022+(updated).xlsx', sheet_name='Data')
plant_df2 = pd.read_excel('../data/Processed_data/steel_plant_matchingresults.xlsx', sheet_name='Sheet1')
merged_df = plant_match(plant_df1, plant_df2)
merged_df.head()

,Plant ID,Plant name (English),Parent,Subnational unit (province/state),Country,Region,Coordinates,Status,Nominal capacity (kt/y),Wiki page,Primary production route,Model Year,Total cost,Unit,correct_plant,equipment_type
0,SBR00001,Gerdau Açominas steel plant,Gerdau S.A.,Minas Gerais,Brazil,Latin America and the Caribbean,"-20.543507, -43.752950",operating,7600,https://www.gem.wiki/Gerdau_A%C3%A7ominas_stee...,BF-BOF,2021,681,$/t,Gerdau Açominas Ouro Branco steel plant,BOF
1,SBR00002,ArcelorMittal Tubarão steel plant,ArcelorMittal,Espirito Santa,Brazil,Latin America and the Caribbean,"-20.253447, -40.242298",operating,7500,https://www.gem.wiki/ArcelorMittal_Tubar%C3%A3...,BF-BOF,2021,695,$/t,ArcelorMittal Tubarão steel plant,BOF
2,SBR00003,ArcelorMittal Monlevade steel plant,ArcelorMittal,Minas Gerais,Brazil,Latin America and the Caribbean,"-19.832129, -43.130164",operating,3500,https://www.gem.wiki/ArcelorMittal_Monlevade_s...,BF-BOF,2021,681,$/t,ArcelorMittal Monlevade steel plant,BOF
3,SBR00004,ArcelorMittal Resende steel plant,ArcelorMittal,Rio de Janeiro,Brazil,Latin America and the Caribbean,"-22.493597, -44.512628",operating,1000,https://www.gem.wiki/ArcelorMittal_Resende_ste...,EAF,2021,748,$/t,ArcelorMittal Resende steel plant,EAF
4,SBR00005,Usiminas Cubatão steel plant,"Techint Group (Ternium/Tenaris) 39.57%, Nippon...",Sao Paulo,Brazil,Latin America and the Caribbean,"-23.863092, -46.376294",mothballed,4000,https://www.gem.wiki/Usiminas_Cubat%C3%A3o_ste...,BF-BOF,2021,700,$/t,Usiminas Cubatão steel plant,BOF


In [28]:
# ====================================Merge steel plant property dataset with GSCT steel plant cost data=============================================================
def steel_property_merge_steel_cost(df1, df2):
    df1.drop_duplicates(inplace = True)
    merged_df = pd.merge(df1, df2[['correct_plant', 'equipment_type', 'Total cost']], left_on=['Plant name (English)', 'Main production equipment'], right_on=['correct_plant', 'equipment_type'], how='left')
    merged_df = merged_df.drop(columns=['correct_plant', 'equipment_type'])
    # fill blank with 0
    merged_df['Total cost'] = merged_df['Total cost'].fillna(0)
    merged_df.drop_duplicates(inplace=True)
    return merged_df

steel_property = pd.read_excel('../data/Processed_data/steel_plant_property_cap_production.xlsx', sheet_name='Sheet1')
steel_prod_cost = pd.read_excel('../data/Processed_data/GSCT_w_globaltrack_plantName.xlsx', sheet_name='Sheet1')
property_cost_merged_df = steel_property_merge_steel_cost(steel_property, steel_prod_cost)
property_cost_merged_df.head()

,GEM plant ID,Plant name (English),Plant name (other language),Owner,Parent (English),Parent GEM entity ID,City,State/Province,Country,Region,...,Primary Owner Country,Secondary Owner Country,Third Owner Country,Fourth Owner Country,Fifth Owner Country,GEM wiki page,Status,capacity_ttpa,production_ttpa,Total cost
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,ABA Çelik Demir LŞ,ABA Çelik Demir LŞ [100.0%],E100000131190 [100.0%],Payas,Hatay,Türkiye,Europe,...,NaN,NaN,NaN,NaN,NaN,https://www.gem.wiki/Aba_Iron_and_Steel_Payas_...,operating,1100,0,0.0
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Abba Steel Ltd,Abba Steel Ltd [100.0%],E100001012072 [100.0%],Oshikango,Ohangwena,Namibia,Africa,...,NaN,NaN,NaN,NaN,NaN,https://www.gem.wiki/Abba_Steel_Ohangwena_stee...,highly probable,3000,0,0.0
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,Abinski Elektrometallurgicheski Zavod LLC,Novostal-M LLC [100.0%],E100000131027 [100.0%],Abinsk,Krasnodar,Russia,Eurasia,...,Russia,NaN,NaN,NaN,NaN,https://www.gem.wiki/Abinsk_Electric_Steel_Works,operating,1600,1600,639.0
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,Abul Khair Steel Ltd,Abul Khair Ltd,E100000131048,Sitakunda,Chittagong,Bangladesh,Asia Pacific,...,Bangladesh,NaN,NaN,NaN,NaN,https://www.gem.wiki/Abul_Khair_Steel_Sitakund...,operating,1400,0,0.0
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,Acciaierie d'italia SpA,Acciaierie d'italia SpA [100.0%],E100001010116 [100.0%],Taranto,Province of Taranto,Italy,Europe,...,Italy,NaN,NaN,NaN,NaN,https://www.gem.wiki/Acciaierie_d'Italia_Taran...,probable,2500,0,0.0


In [29]:
property_cost_merged_df.to_excel('../data/Processed_data/steel_property_cost_merged.xlsx', index=False)
property_cost_merged_df.shape

(2320, 41)